In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import numpy as np

In [3]:
from idinn.sourcing_model import SingleSourcingModel
from idinn.single_controller import BaseStockController
from idinn.demand import UniformDemand

In [7]:
single_sourcing_model = SingleSourcingModel(
    lead_time=2,
    holding_cost=5,
    shortage_cost=495,
    batch_size=32,
    init_inventory=10,
    demand_generator=UniformDemand(low=0, high=4),
)
controller_base = BaseStockController()
# z_star should be 11
controller_base.fit(single_sourcing_model)
print(f"z_star: {controller_base.z_star}")
# Total cost near 29
controller_base.get_total_cost(single_sourcing_model, sourcing_periods=1000).mean()/1000

z_star: 11.0


tensor(29.0736, grad_fn=<DivBackward0>)

In [6]:
single_sourcing_model = SingleSourcingModel(
    lead_time=0,
    holding_cost=5,
    shortage_cost=495,
    batch_size=32,
    init_inventory=10,
    demand_generator=UniformDemand(low=0, high=4),
 )
controller_base = BaseStockController()
# z_star should be 4
controller_base.fit(single_sourcing_model)
print(f"z_star: {controller_base.z_star}")
# Total cost near 10
controller_base.get_total_cost(single_sourcing_model, sourcing_periods=1000).mean()/1000

z_star: 4.0


tensor(10.0834, grad_fn=<DivBackward0>)

In [2]:

from idinn.single_controller import SingleSourcingNeuralController
from idinn.dual_controller import DynamicProgrammingController, CappedDualIndexController, DualSourcingNeuralController
from idinn.sourcing_model import SingleSourcingModel, DualSourcingModel
from idinn.demand import UniformDemand
from torch.utils.tensorboard import SummaryWriter

tensor([50, 30, 10, 30, 20])


In [7]:
single_sourcing_model = SingleSourcingModel(
    lead_time=2,
    holding_cost=5,
    shortage_cost=495,
    batch_size=32,
    init_inventory=10,
    demand_generator=UniformDemand(low=0, high=4),
)
controller_base = BaseStockController()
# z_star should be 11
controller_base.fit(single_sourcing_model)
print(f"z_star: {controller_base.z_star}")
# Total cost near 29
controller_base.get_total_cost(single_sourcing_model, sourcing_periods=1000).mean()/1000

z_star: 11.0


tensor(28.8533, grad_fn=<DivBackward0>)

In [8]:
single_sourcing_model = SingleSourcingModel(
    lead_time=0,
    holding_cost=5,
    shortage_cost=495,
    batch_size=32,
    init_inventory=10,
    demand_generator=UniformDemand(low=0, high=4),
 )
controller_base = BaseStockController()
# z_star should be 4
controller_base.fit(single_sourcing_model)
print(f"z_star: {controller_base.z_star}")
# Total cost near 10
controller_base.get_total_cost(single_sourcing_model, sourcing_periods=1000).mean()/1000

z_star: 4.0


tensor(10.1567, grad_fn=<DivBackward0>)

In [146]:
single_sourcing_model = SingleSourcingModel(
    lead_time=0,
    holding_cost=5,
    shortage_cost=495,
    batch_size=32,
    init_inventory=10,
    demand_generator=UniformDemand(low=1, high=4),
 )

In [147]:
controller_base = BaseStockController()
# z_star should be 4
controller_base.fit(single_sourcing_model)
print(f"z_star: {controller_base.z_star}")
# Total cost near 10
controller_base.get_total_cost(single_sourcing_model, sourcing_periods=1000).mean()/1000

z_star: 4.0


tensor(7.5552, grad_fn=<DivBackward0>)

In [148]:
controller_neural = SingleSourcingNeuralController()
controller_neural.fit(
    single_sourcing_model,
    sourcing_periods=50,
    validation_sourcing_periods=1000,
    epochs=2000,
    tensorboard_writer=SummaryWriter(comment="_single_1"),
    seed=1
)
controller_neural.get_total_cost(single_sourcing_model, sourcing_periods=1000).mean() / 1000

tensor(7.5725, grad_fn=<DivBackward0>)

In [3]:
sourcing_model = DualSourcingModel(
    regular_lead_time=3,
    expedited_lead_time=0,
    regular_order_cost=0,
    expedited_order_cost=20,
    holding_cost=5,
    shortage_cost=495,
    init_inventory=0,
    demand_generator=UniformDemand(low=0, high=4)
)

In [6]:
controller_cdi = CappedDualIndexController()

controller_cdi.fit(
    sourcing_model,
    sourcing_periods=100
)

In [28]:
demand = UniformDemand(low=0, high=4)

In [37]:
a = demand.distribution.arg_constraints["low"]


torch.distributions.constraints._Dependent

In [27]:
demand.distribution.log_prob(torch.tensor(1)).exp()

tensor(0.2000)

In [18]:
controller_cdi.predict(
    current_inventory=3, past_regular_orders=np.array([[1, 1, 1]]), past_expedited_orders=np.array([[0, 0, 0]]), regular_lead_time=3, expedited_lead_time=0
)

(2, 0)

In [ ]:
torch.dis

In [20]:
max_demand = 4
min_demand = 0
support = max_demand - min_demand
demand_prob = dict(
    zip(
        np.arange(min_demand, max_demand + 1),
        np.repeat(1.0 / (support + 1.0), int(support + 1)),
    )
)
demand_prob

{0: 0.2, 1: 0.2, 2: 0.2, 3: 0.2, 4: 0.2}

In [19]:

controller.predict(current_inventory=3, past_regular_orders=[1, 1, 1])

(3, 0)

In [4]:
controller = DynamicProgrammingController()

controller.fit(
    sourcing_model,
    max_iterations=1000000,
    tolerance=1e-6
)

iteration: 100 delta: 0.009350578931446307
iteration: 200 delta: 0.0023492748061642033
iteration: 300 delta: 0.0010458565582283086
iteration: 400 delta: 0.0005887833367026474
iteration: 500 delta: 0.00037700937009077506
iteration: 600 delta: 0.000261899188043202
iteration: 700 delta: 0.00019246147789786505
iteration: 800 delta: 0.00014737959925881228
iteration: 900 delta: 0.00011646423381961313
iteration: 1000 delta: 9.434650070261341e-05
iteration: 1100 delta: 7.797939657905317e-05
iteration: 1200 delta: 6.552931391112793e-05
iteration: 1300 delta: 5.583920487595151e-05
iteration: 1400 delta: 4.8149713059331134e-05
iteration: 1500 delta: 4.1945746040994436e-05
iteration: 1600 delta: 3.686791349366558e-05
iteration: 1700 delta: 3.265928249263084e-05
iteration: 1800 delta: 2.9132225062511452e-05
iteration: 1900 delta: 2.614713785575873e-05
iteration: 2000 delta: 2.359841259291784e-05
iteration: 2100 delta: 2.140496525626645e-05
iteration: 2200 delta: 1.950370641523591e-05
iteration: 230

{(-5, 0, 0): (3, 9),
 (-5, 0, 1): (3, 9),
 (-5, 0, 2): (2, 9),
 (-5, 0, 3): (2, 9),
 (-5, 0, 4): (1, 9),
 (-5, 1, 0): (3, 9),
 (-5, 1, 1): (3, 9),
 (-5, 1, 2): (2, 9),
 (-5, 1, 3): (2, 9),
 (-5, 1, 4): (1, 9),
 (-5, 2, 0): (3, 9),
 (-5, 2, 1): (3, 9),
 (-5, 2, 2): (2, 9),
 (-5, 2, 3): (2, 9),
 (-5, 2, 4): (1, 9),
 (-5, 3, 0): (3, 9),
 (-5, 3, 1): (2, 9),
 (-5, 3, 2): (2, 9),
 (-5, 3, 3): (1, 9),
 (-5, 3, 4): (0, 9),
 (-5, 4, 0): (2, 9),
 (-5, 4, 1): (2, 9),
 (-5, 4, 2): (1, 9),
 (-5, 4, 3): (1, 9),
 (-5, 4, 4): (0, 9),
 (-4, 0, 0): (3, 8),
 (-4, 0, 1): (3, 8),
 (-4, 0, 2): (2, 8),
 (-4, 0, 3): (2, 8),
 (-4, 0, 4): (1, 8),
 (-4, 1, 0): (3, 8),
 (-4, 1, 1): (3, 8),
 (-4, 1, 2): (2, 8),
 (-4, 1, 3): (2, 8),
 (-4, 1, 4): (1, 8),
 (-4, 2, 0): (3, 8),
 (-4, 2, 1): (3, 8),
 (-4, 2, 2): (2, 8),
 (-4, 2, 3): (2, 8),
 (-4, 2, 4): (1, 8),
 (-4, 3, 0): (3, 8),
 (-4, 3, 1): (2, 8),
 (-4, 3, 2): (2, 8),
 (-4, 3, 3): (1, 8),
 (-4, 3, 4): (0, 8),
 (-4, 4, 0): (2, 8),
 (-4, 4, 1): (2, 8),
 (-4, 4, 2): 

In [10]:
controller.predict(current_inventory=2, past_regular_orders=[1, 1, 1])

(3, 1)

In [10]:
current_inventory = 10
past_regular_orders = [1, 2, 3, 4, 5]
past_expedited_orders = [6, 7, 8]
regular_lead_time = 3



(13, 4, 5)


In [21]:
import numpy as np
current_inventory = 10
regular_lead_time = 5
expedited_lead_time = 2
past_regular_orders = np.array([1, 2, 3, 4, 5, 6, 7, 8])
past_expedited_orders = np.array([6, 7, 8])
# Output:
past_orders = [4, 5, 6, 7, 8]

# Pad with zeros for past_expedited_orders
# e_2 = np.pad(e_2, (0, 3), 'constant')
# Start with np.zeros before padding
e_1 = np.zeros(regular_lead_time)
e_1 = past_regular_orders[-regular_lead_time:]

e_2 = np.zeros(regular_lead_time)
e_2[:expedited_lead_time] = past_expedited_orders[-expedited_lead_time:]
e_3 = e_1 + e_2

first = current_inventory + e_3[0]
second = e_3[-regular_lead_time+1:]
print(tuple([first]+second))

(34.0, 27.0, 28.0, 29.0)


In [ ]:
first =  past_regular_orders[-regular_lead_time]
second = past_regular_orders[-regular_lead_time+1:]
print(tuple([first]+second))

In [ ]:
controller = CappedDualIndexController()
controller.fit(sourcing_model, sourcing_periods=1000)
controller.predict(
    current_inventory,
    past_regular_orders,
    past_expedited_orders
)

In [3]:
sourcing_model = DualSourcingModel(
    regular_lead_time=3,
    expedited_lead_time=0,
    regular_order_cost=0,
    expedited_order_cost=20,
    holding_cost=5,
    shortage_cost=495,
    init_inventory=0,
    demand_generator=UniformDemand(low=0, high=4)
)

controller = DynamicProgrammingController()

controller.fit(
    sourcing_model,
    max_iterations=10000,
    tolerance=1
)